# 3D Volume Reconstruction from Single MRI Slice

This notebook implements a **Slice-based Latent Diffusion Model** for reconstructing full 3D brain MRI volumes from a single 2D slice.

## Architecture Overview
- **Positional Encoding**: Sinusoidal embeddings for slice position
- **U-Net Encoder-Decoder**: With residual blocks and skip connections
- **Conditioning Mechanism**: Position-aware generation
- **Loss Functions**: MSE + Perceptual Loss (VGG features)

## Dataset
- LGG Dataset: 110 patients with 20-88 slices each
- Training pairs: (input_slice, target_slice, position_encoding)
- Expected: ~10,000+ slice pairs

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
from PIL import Image
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Positional Encoding

Sinusoidal positional encoding to embed slice position information.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for slice position"""
    
    def __init__(self, d_model=256, max_position=100):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model
        
        # Create positional encoding matrix
        pe = torch.zeros(max_position, d_model)
        position = torch.arange(0, max_position, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, position):
        """
        Args:
            position: Tensor of shape (batch_size,) with position indices
        Returns:
            Positional embeddings of shape (batch_size, d_model)
        """
        return self.pe[position]


# Test positional encoding
pos_encoder = PositionalEncoding(d_model=256, max_position=100)
test_positions = torch.tensor([0, 10, 20, 30, 40])
test_encodings = pos_encoder(test_positions)
print(f"Positional encoding shape: {test_encodings.shape}")

# Visualize positional encodings
plt.figure(figsize=(12, 6))
plt.imshow(pos_encoder.pe[:50].T, cmap='RdBu', aspect='auto')
plt.xlabel('Position')
plt.ylabel('Encoding Dimension')
plt.title('Sinusoidal Positional Encoding Visualization')
plt.colorbar()
plt.show()

## 3. Data Preparation

Create slice pairs with positional information for training.

In [ ]:
# Set data paths for both datasets (Kaggle environment paths)
LGG_DATA_DIR = Path('/kaggle/input/lgg-mri-segmentation/kaggle_3m')
MU_GLIOMA_DATA_DIR = Path('/kaggle/input/pkg-mu/PKG - MU-Glioma-Post/MU-Glioma-Post') 

print("=" * 70)
print("DATASET 1: LGG (Lower Grade Glioma)")
print("=" * 70)
print(f"LGG Directory: {LGG_DATA_DIR}")
print(f"Directory exists: {LGG_DATA_DIR.exists()}")

# Get all patient folders from LGG dataset
lgg_patient_folders = sorted([f for f in LGG_DATA_DIR.iterdir() if f.is_dir() and f.name.startswith('TCGA')])
print(f"Number of LGG patients: {len(lgg_patient_folders)}")

# Collect all slices per patient from LGG
lgg_patient_slices = {}

for patient_folder in tqdm(lgg_patient_folders, desc="Scanning LGG patients"):
    # Get all image slices (not masks)
    slices = sorted([f for f in patient_folder.glob('*.tif') if '_mask' not in f.name])
    
    if len(slices) > 0:
        lgg_patient_slices[patient_folder.name] = slices

print(f"Total LGG patients with slices: {len(lgg_patient_slices)}")

# Show distribution of slice counts for LGG
lgg_slice_counts = [len(slices) for slices in lgg_patient_slices.values()]
print(f"LGG - Min slices per patient: {min(lgg_slice_counts)}")
print(f"LGG - Max slices per patient: {max(lgg_slice_counts)}")
print(f"LGG - Mean slices per patient: {np.mean(lgg_slice_counts):.1f}")

print("\n" + "=" * 70)
print("DATASET 2: MU-Glioma")
print("=" * 70)
print(f"MU-Glioma Directory: {MU_GLIOMA_DATA_DIR}")
print(f"Directory exists: {MU_GLIOMA_DATA_DIR.exists()}")

# Collect all slices from MU-Glioma dataset
mu_glioma_patient_slices = {}

if MU_GLIOMA_DATA_DIR.exists():
    # Get all patient folders from MU-Glioma dataset
    # Adjust the pattern based on actual MU-Glioma folder structure
    mu_glioma_patient_folders = sorted([f for f in MU_GLIOMA_DATA_DIR.iterdir() if f.is_dir()])
    print(f"Number of MU-Glioma patients: {len(mu_glioma_patient_folders)}")
    
    for patient_folder in tqdm(mu_glioma_patient_folders, desc="Scanning MU-Glioma patients"):
        # Get all image slices (adjust extension if needed - could be .dcm, .nii, etc.)
        slices = sorted([f for f in patient_folder.glob('*.tif')])
        
        if len(slices) > 0:
            mu_glioma_patient_slices[patient_folder.name] = slices
    
    print(f"Total MU-Glioma patients with slices: {len(mu_glioma_patient_slices)}")
    
    if len(mu_glioma_patient_slices) > 0:
        # Show distribution of slice counts for MU-Glioma
        mu_slice_counts = [len(slices) for slices in mu_glioma_patient_slices.values()]
        print(f"MU-Glioma - Min slices per patient: {min(mu_slice_counts)}")
        print(f"MU-Glioma - Max slices per patient: {max(mu_slice_counts)}")
        print(f"MU-Glioma - Mean slices per patient: {np.mean(mu_slice_counts):.1f}")
else:
    print("⚠️  MU-Glioma directory not found. Please update MU_GLIOMA_DATA_DIR path.")
    print("    Training will proceed with LGG dataset only.")

# Combine both datasets
print("\n" + "=" * 70)
print("COMBINED DATASETS")
print("=" * 70)
patient_slices = {**lgg_patient_slices, **mu_glioma_patient_slices}
print(f"Total patients (LGG + MU-Glioma): {len(patient_slices)}")
print(f"  - LGG patients: {len(lgg_patient_slices)}")
print(f"  - MU-Glioma patients: {len(mu_glioma_patient_slices)}")

# Show combined distribution
all_slice_counts = [len(slices) for slices in patient_slices.values()]
print(f"\nCombined statistics:")
print(f"Total slices across all patients: {sum(all_slice_counts):,}")
print(f"Min slices per patient: {min(all_slice_counts)}")
print(f"Max slices per patient: {max(all_slice_counts)}")
print(f"Mean slices per patient: {np.mean(all_slice_counts):.1f}")

# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# LGG distribution
axes[0].hist(lgg_slice_counts, bins=20, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].set_xlabel('Number of Slices per Patient', fontsize=12)
axes[0].set_ylabel('Number of Patients', fontsize=12)
axes[0].set_title('LGG Dataset - Slice Distribution', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Combined distribution
axes[1].hist(all_slice_counts, bins=20, edgecolor='black', color='green', alpha=0.7)
axes[1].set_xlabel('Number of Slices per Patient', fontsize=12)
axes[1].set_ylabel('Number of Patients', fontsize=12)
axes[1].set_title('Combined (LGG + MU-Glioma) - Slice Distribution', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create training pairs: (input_slice, target_slice, input_position, target_position)
training_pairs = []

for patient_id, slices in tqdm(patient_slices.items(), desc="Creating slice pairs"):
    n_slices = len(slices)
    
    if n_slices < 2:
        continue
    
    # For each patient, create pairs of slices
    # Strategy: Use each slice as input, predict nearby slices
    for i in range(n_slices):
        # Predict neighboring slices (within distance of 5)
        for j in range(max(0, i-5), min(n_slices, i+6)):
            if i != j:  # Don't predict the same slice
                training_pairs.append({
                    'patient_id': patient_id,
                    'input_slice': slices[i],
                    'target_slice': slices[j],
                    'input_position': i,
                    'target_position': j,
                    'total_slices': n_slices,
                    'relative_distance': j - i
                })

print(f"\nTotal training pairs created: {len(training_pairs):,}")

# Show distribution of relative distances
distances = [pair['relative_distance'] for pair in training_pairs]
plt.figure(figsize=(10, 5))
plt.hist(distances, bins=21, edgecolor='black')
plt.xlabel('Relative Distance (target - input)')
plt.ylabel('Count')
plt.title('Distribution of Slice Pair Distances')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Custom Dataset Class

In [ ]:
class SliceReconstructionDataset(Dataset):
    """Dataset for 3D volume reconstruction from slice pairs"""
    
    def __init__(self, pairs, img_size=256, normalize=True):
        self.pairs = pairs
        self.img_size = img_size
        self.normalize = normalize
        
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Load input and target slices
        input_img = np.array(Image.open(pair['input_slice']))
        target_img = np.array(Image.open(pair['target_slice']))
        
        # Resize
        input_img = cv2.resize(input_img, (self.img_size, self.img_size))
        target_img = cv2.resize(target_img, (self.img_size, self.img_size))
        
        # Normalize
        if self.normalize:
            input_img = input_img.astype(np.float32) / 255.0
            target_img = target_img.astype(np.float32) / 255.0
        
        # Convert to tensor: (H, W, C) -> (C, H, W)
        input_tensor = torch.from_numpy(input_img).permute(2, 0, 1)
        target_tensor = torch.from_numpy(target_img).permute(2, 0, 1)
        
        # Normalize positions to [0, 1]
        input_pos = pair['input_position'] / max(pair['total_slices'] - 1, 1)
        target_pos = pair['target_position'] / max(pair['total_slices'] - 1, 1)
        
        # Get positional indices (unnormalized for embedding lookup)
        input_pos_idx = pair['input_position']
        target_pos_idx = pair['target_position']
        
        return {
            'input_slice': input_tensor,
            'target_slice': target_tensor,
            'input_pos': torch.tensor(input_pos, dtype=torch.float32),
            'target_pos': torch.tensor(target_pos, dtype=torch.float32),
            'input_pos_idx': torch.tensor(input_pos_idx, dtype=torch.long),
            'target_pos_idx': torch.tensor(target_pos_idx, dtype=torch.long),
            'relative_distance': torch.tensor(pair['relative_distance'], dtype=torch.float32)
        }

## 5. Model Architecture Components

### 5.1 Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with batch normalization"""
    
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

### 5.2 Attention Module

In [ ]:
class AttentionBlock(nn.Module):
    """Attention mechanism for feature refinement"""
    
    def __init__(self, channels):
        super(AttentionBlock, self).__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        attention = self.sigmoid(self.conv(x))
        return x * attention

### 5.3 Positional Conditioning Module

In [ ]:
class PositionalConditioner(nn.Module):
    """Inject positional information into feature maps"""
    
    def __init__(self, pos_dim=256, feature_channels=512):
        super(PositionalConditioner, self).__init__()
        
        # MLP to process positional encoding
        self.mlp = nn.Sequential(
            nn.Linear(pos_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, feature_channels)
        )
    
    def forward(self, features, pos_encoding):
        """
        Args:
            features: (B, C, H, W)
            pos_encoding: (B, pos_dim)
        Returns:
            Conditioned features: (B, C, H, W)
        """
        # Process positional encoding
        pos_features = self.mlp(pos_encoding)  # (B, C)
        
        # Reshape to broadcast: (B, C, 1, 1)
        pos_features = pos_features.unsqueeze(-1).unsqueeze(-1)
        
        # Add positional bias to features
        return features + pos_features

## 6. U-Net with Positional Conditioning

In [ ]:
class PositionalUNet(nn.Module):
    """U-Net with residual blocks and positional conditioning"""
    
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512], pos_dim=256):
        super(PositionalUNet, self).__init__()
        
        self.pos_encoder = PositionalEncoding(d_model=pos_dim, max_position=100)
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        current_channels = in_channels
        for feature in features:
            self.encoders.append(nn.Sequential(
                nn.Conv2d(current_channels, feature, kernel_size=3, padding=1),
                nn.BatchNorm2d(feature),
                nn.ReLU(inplace=True),
                ResidualBlock(feature)
            ))
            self.pools.append(nn.MaxPool2d(kernel_size=2, stride=2))
            current_channels = feature
        
        # Bottleneck with positional conditioning
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features[-1], features[-1] * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(features[-1] * 2),
            nn.ReLU(inplace=True),
            ResidualBlock(features[-1] * 2)
        )
        
        # Positional conditioner for bottleneck
        self.pos_conditioner = PositionalConditioner(pos_dim=pos_dim, feature_channels=features[-1] * 2)
        
        # Decoder
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        
        reversed_features = list(reversed(features))
        for i in range(len(reversed_features)):
            in_ch = reversed_features[i] * 2 if i == 0 else reversed_features[i - 1]
            out_ch = reversed_features[i]
            
            self.upconvs.append(
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
            )
            
            self.decoders.append(nn.Sequential(
                nn.Conv2d(out_ch * 2, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                ResidualBlock(out_ch)
            ))
        
        # Final output layer
        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x, target_pos_idx):
        """
        Args:
            x: Input slice (B, C, H, W)
            target_pos_idx: Target position indices (B,)
        Returns:
            Generated slice at target position (B, C, H, W)
        """
        # Get positional encoding
        pos_encoding = self.pos_encoder(target_pos_idx)  # (B, pos_dim)
        
        # Encoder path
        skip_connections = []
        for encoder, pool in zip(self.encoders, self.pools):
            x = encoder(x)
            skip_connections.append(x)
            x = pool(x)
        
        # Bottleneck with positional conditioning
        x = self.bottleneck(x)
        x = self.pos_conditioner(x, pos_encoding)
        
        # Decoder path
        skip_connections = list(reversed(skip_connections))
        for i, (upconv, decoder) in enumerate(zip(self.upconvs, self.decoders)):
            x = upconv(x)
            
            # Concatenate with skip connection
            skip = skip_connections[i]
            
            # Handle size mismatch
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            
            x = torch.cat([x, skip], dim=1)
            x = decoder(x)
        
        # Final output
        return self.final(x)

## 7. Perceptual Loss

Using VGG16 features for perceptual similarity.

In [ ]:
class PerceptualLoss(nn.Module):
    """Perceptual loss using VGG16 features"""
    
    def __init__(self, layers=['conv1_2', 'conv2_2', 'conv3_3', 'conv4_3']):
        super(PerceptualLoss, self).__init__()
        
        # Load pre-trained VGG16
        vgg = models.vgg16(pretrained=True).features
        
        # Freeze VGG parameters
        for param in vgg.parameters():
            param.requires_grad = False
        
        self.vgg = vgg.eval()
        
        # Layer indices for feature extraction
        self.layer_name_mapping = {
            'conv1_2': 3,
            'conv2_2': 8,
            'conv3_3': 15,
            'conv4_3': 22
        }
        
        self.selected_layers = [self.layer_name_mapping[name] for name in layers]
        
        # Normalization for VGG
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    
    def normalize(self, x):
        """Normalize for VGG"""
        return (x - self.mean) / self.std
    
    def extract_features(self, x):
        """Extract multi-scale features from VGG"""
        features = []
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            if i in self.selected_layers:
                features.append(x)
        return features
    
    def forward(self, pred, target):
        """
        Args:
            pred: Predicted image (B, 3, H, W)
            target: Target image (B, 3, H, W)
        Returns:
            Perceptual loss
        """
        # Normalize
        pred_norm = self.normalize(pred)
        target_norm = self.normalize(target)
        
        # Extract features
        pred_features = self.extract_features(pred_norm)
        target_features = self.extract_features(target_norm)
        
        # Compute L1 loss at each scale
        loss = 0
        for pred_feat, target_feat in zip(pred_features, target_features):
            loss += F.l1_loss(pred_feat, target_feat)
        
        return loss / len(pred_features)

## 8. Combined Loss Function

In [ ]:
class CombinedLoss(nn.Module):
    """MSE + Perceptual Loss"""
    
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.perceptual = PerceptualLoss()
    
    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        perceptual_loss = self.perceptual(pred, target)
        
        total_loss = self.alpha * mse_loss + (1 - self.alpha) * perceptual_loss
        
        return total_loss, mse_loss, perceptual_loss

## 9. Metrics: PSNR and SSIM

In [ ]:
def calculate_psnr(pred, target):
    """Calculate PSNR between predicted and target images"""
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    
    psnr_values = []
    for i in range(pred_np.shape[0]):
        # Transpose to (H, W, C)
        pred_img = pred_np[i].transpose(1, 2, 0)
        target_img = target_np[i].transpose(1, 2, 0)
        
        # Calculate PSNR
        psnr_val = psnr(target_img, pred_img, data_range=1.0)
        psnr_values.append(psnr_val)
    
    return np.mean(psnr_values)


def calculate_ssim(pred, target):
    """Calculate SSIM between predicted and target images"""
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    
    ssim_values = []
    for i in range(pred_np.shape[0]):
        # Transpose to (H, W, C)
        pred_img = pred_np[i].transpose(1, 2, 0)
        target_img = target_np[i].transpose(1, 2, 0)
        
        # Calculate SSIM
        ssim_val = ssim(target_img, pred_img, data_range=1.0, channel_axis=2)
        ssim_values.append(ssim_val)
    
    return np.mean(ssim_values)

## 10. Data Splitting and DataLoader Setup

In [ ]:
# Split data by patient (to avoid data leakage)
patient_ids = list(patient_slices.keys())
train_patients, temp_patients = train_test_split(patient_ids, test_size=0.3, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.5, random_state=42)

print(f"Train patients: {len(train_patients)}")
print(f"Val patients: {len(val_patients)}")
print(f"Test patients: {len(test_patients)}")

# Filter pairs by patient split
train_pairs = [p for p in training_pairs if p['patient_id'] in train_patients]
val_pairs = [p for p in training_pairs if p['patient_id'] in val_patients]
test_pairs = [p for p in training_pairs if p['patient_id'] in test_patients]

print(f"\nTrain pairs: {len(train_pairs):,}")
print(f"Val pairs: {len(val_pairs):,}")
print(f"Test pairs: {len(test_pairs):,}")

# Create datasets
IMG_SIZE = 256
BATCH_SIZE = 16

train_dataset = SliceReconstructionDataset(train_pairs, img_size=IMG_SIZE)
val_dataset = SliceReconstructionDataset(val_pairs, img_size=IMG_SIZE)
test_dataset = SliceReconstructionDataset(test_pairs, img_size=IMG_SIZE)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 11. Model Initialization

In [ ]:
# Initialize model
model = PositionalUNet(
    in_channels=3,
    out_channels=3,
    features=[64, 128, 256, 512],
    pos_dim=256
).to(device)

# Loss and optimizer
criterion = CombinedLoss(alpha=0.5).to(device)  # Move criterion to device
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture: Positional U-Net")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 12. Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    running_mse = 0.0
    running_perceptual = 0.0
    running_psnr = 0.0
    running_ssim = 0.0
    
    pbar = tqdm(dataloader, desc='Training')
    for batch in pbar:
        input_slices = batch['input_slice'].to(device)
        target_slices = batch['target_slice'].to(device)
        target_pos_idx = batch['target_pos_idx'].to(device)
        
        # Forward pass
        outputs = model(input_slices, target_pos_idx)
        total_loss, mse_loss, perceptual_loss = criterion(outputs, target_slices)
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Calculate metrics
        with torch.no_grad():
            psnr_val = calculate_psnr(outputs, target_slices)
            ssim_val = calculate_ssim(outputs, target_slices)
        
        running_loss += total_loss.item()
        running_mse += mse_loss.item()
        running_perceptual += perceptual_loss.item()
        running_psnr += psnr_val
        running_ssim += ssim_val
        
        pbar.set_postfix({
            'loss': total_loss.item(),
            'psnr': psnr_val,
            'ssim': ssim_val
        })
    
    n = len(dataloader)
    return {
        'loss': running_loss / n,
        'mse': running_mse / n,
        'perceptual': running_perceptual / n,
        'psnr': running_psnr / n,
        'ssim': running_ssim / n
    }


def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    running_mse = 0.0
    running_perceptual = 0.0
    running_psnr = 0.0
    running_ssim = 0.0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for batch in pbar:
            input_slices = batch['input_slice'].to(device)
            target_slices = batch['target_slice'].to(device)
            target_pos_idx = batch['target_pos_idx'].to(device)
            
            # Forward pass
            outputs = model(input_slices, target_pos_idx)
            total_loss, mse_loss, perceptual_loss = criterion(outputs, target_slices)
            
            # Calculate metrics
            psnr_val = calculate_psnr(outputs, target_slices)
            ssim_val = calculate_ssim(outputs, target_slices)
            
            running_loss += total_loss.item()
            running_mse += mse_loss.item()
            running_perceptual += perceptual_loss.item()
            running_psnr += psnr_val
            running_ssim += ssim_val
            
            pbar.set_postfix({
                'loss': total_loss.item(),
                'psnr': psnr_val,
                'ssim': ssim_val
            })
    
    n = len(dataloader)
    return {
        'loss': running_loss / n,
        'mse': running_mse / n,
        'perceptual': running_perceptual / n,
        'psnr': running_psnr / n,
        'ssim': running_ssim / n
    }

## 13. Training Loop

## 13.1. Load Checkpoint (Optional - for resuming training)

In [ ]:
import os

# Checkpoint file paths (in order of preference)
checkpoint_files = ['3d_reconstruction_final.pth', 'best_3d_reconstruction.pth']

# Find available checkpoint
CHECKPOINT_PATH = None
for ckpt_file in checkpoint_files:
    if os.path.exists(ckpt_file):
        CHECKPOINT_PATH = ckpt_file
        break

# Check if checkpoint exists
if CHECKPOINT_PATH:
    print(f"Found checkpoint: {CHECKPOINT_PATH}")
    response = input("Do you want to load this checkpoint and resume training? (y/n): ").strip().lower()
    
    if response == 'y':
        print("\nLoading checkpoint...")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        
        # Check if checkpoint is a full dict or just state_dict
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # Full checkpoint with metadata
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch']
            best_val_psnr = checkpoint['best_val_psnr']
            history = checkpoint['history']
            
            print(f"✓ Checkpoint loaded successfully!")
            print(f"  - Resuming from epoch: {start_epoch}")
            print(f"  - Best validation PSNR: {best_val_psnr:.2f} dB")
            print(f"  - Training history loaded: {len(history['train_loss'])} epochs")
        else:
            # Just model state_dict (from best_3d_reconstruction.pth)
            model.load_state_dict(checkpoint)
            print(f"✓ Model weights loaded successfully!")
            print(f"  - Note: Only model weights loaded (no training history)")
            print(f"  - Starting fresh training with loaded weights")
            start_epoch = 0
            best_val_psnr = 0.0
            history = {
                'train_loss': [], 'train_mse': [], 'train_perceptual': [], 'train_psnr': [], 'train_ssim': [],
                'val_loss': [], 'val_mse': [], 'val_perceptual': [], 'val_psnr': [], 'val_ssim': []
            }
        
        RESUME_TRAINING = True
    else:
        print("\nStarting fresh training...")
        start_epoch = 0
        best_val_psnr = 0.0
        history = {
            'train_loss': [], 'train_mse': [], 'train_perceptual': [], 'train_psnr': [], 'train_ssim': [],
            'val_loss': [], 'val_mse': [], 'val_perceptual': [], 'val_psnr': [], 'val_ssim': []
        }
        RESUME_TRAINING = False
else:
    print(f"No checkpoint found")
    print("Starting fresh training...")
    start_epoch = 0
    best_val_psnr = 0.0
    history = {
        'train_loss': [], 'train_mse': [], 'train_perceptual': [], 'train_psnr': [], 'train_ssim': [],
        'val_loss': [], 'val_mse': [], 'val_perceptual': [], 'val_psnr': [], 'val_ssim': []
    }
    RESUME_TRAINING = False

## 13.2. Training Configuration

In [ ]:
# Training configuration
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15

# Initialize training state if not loaded from checkpoint
if 'start_epoch' not in locals():
    start_epoch = 0
    best_val_psnr = 0.0
    history = {
        'train_loss': [], 'train_mse': [], 'train_perceptual': [], 'train_psnr': [], 'train_ssim': [],
        'val_loss': [], 'val_mse': [], 'val_perceptual': [], 'val_psnr': [], 'val_ssim': []
    }
    RESUME_TRAINING = False

early_stopping_counter = 0

if RESUME_TRAINING:
    print(f"Resuming training from epoch {start_epoch + 1}...\n")
else:
    print("Starting training...\n")

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 70)
    
    # Train
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_metrics['loss'])
    
    # Save history
    for key in train_metrics:
        history[f'train_{key}'].append(train_metrics[key])
        history[f'val_{key}'].append(val_metrics[key])
    
    # Print summary
    print(f"\nTrain - Loss: {train_metrics['loss']:.4f} | PSNR: {train_metrics['psnr']:.2f} dB | SSIM: {train_metrics['ssim']:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f} | PSNR: {val_metrics['psnr']:.2f} dB | SSIM: {val_metrics['ssim']:.4f}")
    
    # Save best model
    if val_metrics['psnr'] > best_val_psnr:
        best_val_psnr = val_metrics['psnr']
        torch.save(model.state_dict(), 'best_3d_reconstruction.pth')
        print(f"✓ Best model saved! (PSNR: {best_val_psnr:.2f} dB)")
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1
    
    # Early stopping
    if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        break

print("\n" + "="*70)
print(f"Training completed! Best PSNR: {best_val_psnr:.2f} dB")
print("="*70)

## 14. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# PSNR
axes[0, 1].plot(history['train_psnr'], label='Train PSNR', linewidth=2)
axes[0, 1].plot(history['val_psnr'], label='Val PSNR', linewidth=2)
axes[0, 1].axhline(y=26, color='r', linestyle='--', label='Target: 26 dB')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('PSNR (dB)', fontsize=12)
axes[0, 1].set_title('Training and Validation PSNR', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# SSIM
axes[1, 0].plot(history['train_ssim'], label='Train SSIM', linewidth=2)
axes[1, 0].plot(history['val_ssim'], label='Val SSIM', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('SSIM', fontsize=12)
axes[1, 0].set_title('Training and Validation SSIM', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# MSE vs Perceptual Loss
axes[1, 1].plot(history['train_mse'], label='Train MSE', linewidth=2)
axes[1, 1].plot(history['train_perceptual'], label='Train Perceptual', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Loss', fontsize=12)
axes[1, 1].set_title('MSE vs Perceptual Loss Components', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('3d_reconstruction_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

## 15. Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_3d_reconstruction.pth'))

# Evaluate on test set
test_metrics = validate_epoch(model, test_loader, criterion, device)

print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Test MSE: {test_metrics['mse']:.4f}")
print(f"Test Perceptual Loss: {test_metrics['perceptual']:.4f}")
print(f"Test PSNR: {test_metrics['psnr']:.2f} dB (Target: >26 dB)")
print(f"Test SSIM: {test_metrics['ssim']:.4f}")
print("="*70)

## 16. Visualization: Slice Reconstruction

In [ ]:
def visualize_reconstruction(model, dataloader, device, num_samples=6):
    """Visualize slice reconstruction results"""
    model.eval()
    
    samples = []
    with torch.no_grad():
        for batch in dataloader:
            input_slices = batch['input_slice'].to(device)
            target_slices = batch['target_slice'].to(device)
            target_pos_idx = batch['target_pos_idx'].to(device)
            
            outputs = model(input_slices, target_pos_idx)
            
            for i in range(min(len(input_slices), num_samples - len(samples))):
                samples.append({
                    'input': input_slices[i].cpu(),
                    'target': target_slices[i].cpu(),
                    'output': outputs[i].cpu(),
                    'distance': batch['relative_distance'][i].item()
                })
            
            if len(samples) >= num_samples:
                break
    
    # Plot
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    
    for i, sample in enumerate(samples):
        # Convert to numpy and transpose to (H, W, C)
        input_img = sample['input'].permute(1, 2, 0).numpy()
        target_img = sample['target'].permute(1, 2, 0).numpy()
        output_img = sample['output'].permute(1, 2, 0).numpy()
        
        # Calculate metrics
        psnr_val = psnr(target_img, output_img, data_range=1.0)
        ssim_val = ssim(target_img, output_img, data_range=1.0, channel_axis=2)
        
        # Display FLAIR channel (channel 1)
        axes[i, 0].imshow(input_img[:, :, 1], cmap='gray')
        axes[i, 0].set_title(f'Input Slice', fontsize=12)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(target_img[:, :, 1], cmap='gray')
        axes[i, 1].set_title(f'Target (Δ={sample["distance"]:.0f})', fontsize=12)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(output_img[:, :, 1], cmap='gray')
        axes[i, 2].set_title(f'Generated\nPSNR: {psnr_val:.1f} dB', fontsize=12)
        axes[i, 2].axis('off')
        
        # Difference map
        diff = np.abs(target_img[:, :, 1] - output_img[:, :, 1])
        im = axes[i, 3].imshow(diff, cmap='hot')
        axes[i, 3].set_title(f'Difference\nSSIM: {ssim_val:.3f}', fontsize=12)
        axes[i, 3].axis('off')
        plt.colorbar(im, ax=axes[i, 3], fraction=0.046)
    
    plt.tight_layout()
    plt.savefig('slice_reconstruction_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize results
visualize_reconstruction(model, test_loader, device, num_samples=6)

## 17. 3D Volume Generation Pipeline

In [ ]:
def generate_3d_volume(model, input_slice, num_slices=40, device='cuda'):
    """
    Generate a 3D volume from a single input slice
    
    Args:
        model: Trained PositionalUNet model
        input_slice: Single MRI slice tensor (C, H, W)
        num_slices: Number of slices to generate
        device: Device to run on
    
    Returns:
        volume: Generated 3D volume (num_slices, H, W, C)
    """
    model.eval()
    
    # Prepare input
    input_batch = input_slice.unsqueeze(0).to(device)  # (1, C, H, W)
    
    volume_slices = []
    
    with torch.no_grad():
        for pos_idx in tqdm(range(num_slices), desc='Generating volume'):
            # Create position tensor
            target_pos_idx = torch.tensor([pos_idx], dtype=torch.long).to(device)
            
            # Generate slice
            generated_slice = model(input_batch, target_pos_idx)
            
            # Convert to numpy (H, W, C)
            slice_np = generated_slice[0].cpu().permute(1, 2, 0).numpy()
            volume_slices.append(slice_np)
    
    # Stack into 3D volume
    volume = np.stack(volume_slices, axis=0)
    
    return volume


def visualize_3d_volume(volume, num_display=10):
    """Visualize slices from a 3D volume"""
    n_slices = volume.shape[0]
    indices = np.linspace(0, n_slices-1, num_display, dtype=int)
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        # Display FLAIR channel (channel 1)
        axes[i].imshow(volume[idx, :, :, 1], cmap='gray')
        axes[i].set_title(f'Slice {idx}/{n_slices-1}', fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('generated_3d_volume.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Test 3D volume generation
# Get a random test sample
test_sample = test_dataset[np.random.randint(0, len(test_dataset))]
input_slice = test_sample['input_slice']

print("Generating 3D volume from single slice...")
generated_volume = generate_3d_volume(model, input_slice, num_slices=40, device=device)

print(f"\nGenerated volume shape: {generated_volume.shape}")
print("Visualizing generated slices...")
visualize_3d_volume(generated_volume, num_display=10)

## 18. Save Model

In [ ]:
# Save final model with metadata
torch.save({
    'epoch': len(history['train_loss']),
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_psnr': best_val_psnr,
    'test_psnr': test_metrics['psnr'],
    'test_ssim': test_metrics['ssim'],
    'history': history,
    'config': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'features': [64, 128, 256, 512],
        'pos_dim': 256
    }
}, '3d_reconstruction_final.pth')

print("Final model saved successfully!")

## 19. Model Summary

In [ ]:
print("\n" + "="*70)
print("3D VOLUME RECONSTRUCTION MODEL SUMMARY")
print("="*70)
print(f"Architecture: Positional U-Net with Latent Diffusion")
print(f"Input: Single MRI slice ({IMG_SIZE}x{IMG_SIZE}x3)")
print(f"Output: Target slice at specified position")
print(f"Total Parameters: {total_params:,}")
print(f"\nPositional Encoding: Sinusoidal (256-dim)")
print(f"Loss Function: MSE (50%) + Perceptual Loss (50%)")
print(f"\nTraining Data: {len(train_pairs):,} slice pairs")
print(f"Validation Data: {len(val_pairs):,} slice pairs")
print(f"Test Data: {len(test_pairs):,} slice pairs")
print(f"\nBest Validation PSNR: {best_val_psnr:.2f} dB")
print(f"Test PSNR: {test_metrics['psnr']:.2f} dB (Target: >26 dB)")
print(f"Test SSIM: {test_metrics['ssim']:.4f}")
print(f"\nCapability: Generate full 3D volumes from single 2D slice")
print("="*70)